# recon3d: unseen-category evaluation + OOD detector (Kaggle, GPU T4)

This notebook contains **no model code**. It installs the `recon3d` package from your GitHub repo and runs two commands:

1. `recon3d evaluate`: re-evaluates your trained model on the seen test set and compares the numbers with the
   notebook-era results (**parity check**: proves the package reproduces the notebook).
2. `recon3d unseen`: evaluates on 6 never-seen categories and builds `ood_detector.npz` for the web app.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**.
**Inputs (Add Input):**
- Datasets: `recon3d-shapenet6-v2` and `recon3d-unseen6-v1`
- Notebooks: your training notebook (the version whose output has `runs/finetune_resnet18_n1000/best.pt`)

Takes about 20-30 minutes. Use **Save Version -> Save & Run All** so it runs in the background.

In [ ]:
# ── CELL 1 · Install the package from GitHub ───────────────────────────────
import os, sys, glob, json, shutil, subprocess
E = os.environ.get
GITHUB_REPO = "https://github.com/YOUR_GITHUB_USERNAME/recon3d"      # <- EDIT (repo must be public)
GIT_REF     = E("R3D_GIT_REF", "main")                               # a branch, tag or commit hash
RUN         = E("R3D_RUN", "finetune_resnet18_n1000")
INPUT_ROOT  = E("R3D_INPUT", "/kaggle/input")
WORK        = E("R3D_WORK", "/kaggle/working")
PIP_SPEC    = E("R3D_PIP_SPEC", f"git+{GITHUB_REPO}.git@{GIT_REF}")
assert "YOUR_GITHUB_USERNAME" not in PIP_SPEC, "set GITHUB_REPO in this cell"

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIP_SPEC], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip install failed: is the repo public and the URL right?"
out = subprocess.run(["recon3d", "--help"], capture_output=True, text=True)
assert out.returncode == 0, out.stderr
import recon3d, torch
print("recon3d", recon3d.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# ── CELL 2 · Find the inputs ───────────────────────────────────────────────
def datasets():
    for p in glob.glob(f"{INPUT_ROOT}/**/meta.json", recursive=True):
        d = os.path.dirname(p)
        yield d, json.load(open(p))

seen   = [d for d, m in datasets() if m.get("role") != "unseen_categories" and m["counts"].get("train", 0) > 0]
unseen = [d for d, m in datasets() if m.get("role") == "unseen_categories"]
ckpts  = glob.glob(f"{INPUT_ROOT}/**/runs/{RUN}/best.pt", recursive=True)
assert len(seen) == 1, f"need exactly one training dataset attached, found {seen}"
assert len(unseen) == 1, f"need exactly one unseen dataset attached, found {unseen}"
assert ckpts, f"best.pt for {RUN} not found: attach the training notebook's output"
SEEN, UNSEEN = seen[0], unseen[0]

RUNS = os.path.join(WORK, "runs")
run_dir = os.path.join(RUNS, RUN)
os.makedirs(run_dir, exist_ok=True)
src = os.path.dirname(ckpts[0])
shutil.copyfile(os.path.join(src, "best.pt"), os.path.join(run_dir, "best.pt"))
OLD_SUMMARY = os.path.join(WORK, "notebook_eval_test_summary.json")
if os.path.exists(os.path.join(src, "eval_test_summary.json")):
    shutil.copyfile(os.path.join(src, "eval_test_summary.json"), OLD_SUMMARY)
print("seen:  ", SEEN, json.load(open(f"{SEEN}/meta.json"))["counts"])
print("unseen:", UNSEEN, json.load(open(f"{UNSEEN}/meta.json"))["counts"], json.load(open(f"{UNSEEN}/meta.json"))["categories"])
print("checkpoint:", ckpts[0])

In [ ]:
# ── CELL 3 · Parity check: package vs notebook numbers on the seen test set ─
def run(cmd):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"command failed with exit code {p.returncode}"

run(["recon3d", "evaluate", "--data", SEEN, "--runs", RUNS, "--run", RUN, "--splits", "test"])

if os.path.exists(OLD_SUMMARY):
    old = json.load(open(OLD_SUMMARY))["overall"]
    new = json.load(open(os.path.join(run_dir, "eval_test_summary.json")))["overall"]
    print(f"\n{'method':<10} {'metric':<9} {'notebook':>10} {'package':>10} {'diff':>9}")
    worst = 0.0
    for method in old:
        for k in old[method]:
            d = new[method][k] - old[method][k]
            worst = max(worst, abs(d) / max(abs(old[method][k]), 1e-9))
            print(f"{method:<10} {k:<9} {old[method][k]:>10.4f} {new[method][k]:>10.4f} {d:>+9.4f}")
    print(f"\nlargest relative difference: {worst:.2%}  ->  "
          + ("PARITY OK" if worst < 0.01 else "CHECK: more than 1% different"))
else:
    print("no notebook summary found next to best.pt: parity comparison skipped")

In [ ]:
# ── CELL 4 · Unseen categories + OOD detector ──────────────────────────────
run(["recon3d", "unseen", "--data", SEEN, "--unseen", UNSEEN, "--runs", RUNS, "--run", RUN])

In [ ]:
# ── CELL 5 · Show the results ──────────────────────────────────────────────
from IPython.display import Markdown, Image, display
U = os.path.join(run_dir, "unseen")
display(Markdown(open(os.path.join(U, "findings.md")).read()))
for f in ["category_cd.png", "ood_hist.png", "score_vs_cd.png", "qualitative_unseen.png"]:
    p = os.path.join(U, f)
    if os.path.exists(p):
        print(f); display(Image(p))
print("\nFiles to download from Output:", sorted(os.listdir(U)))